In [27]:
import os
import json
import shutil
import numpy as np
from pathlib import Path
from collections import Counter

USER = os.environ.get('USER', 'sigursteinsson1')

# Source: We can just reuse the checkpoint 2 data
TRAINING_DATA_DIR = Path(f"/p/project1/training2600/{USER}/training_data_checkpoint")

# Destination: TerraTorch-compatible folder layout
FINAL_PROJECT_DIR = Path(f"/p/project1/training2600/{USER}/Final_Project")
PROCESSED_DIR     = FINAL_PROJECT_DIR / "processed_data"

CORINE_CLASSES = {
    1: "continuous_urban_fabric", 2: "discontinuous_urban_fabric",
    3: "industrial_or_commercial_units", 4: "road_and_rail_networks",
    5: "port_areas", 6: "airports", 7: "mineral_extraction_sites",
    8: "dump_sites", 9: "construction_sites", 10: "green_urban_areas",
    11: "sport_and_leisure_facilities", 12: "non_irrigated_arable_land",
    13: "permanently_irrigated_land", 14: "rice_fields", 15: "vineyards",
    16: "fruit_trees_and_berry_plantations", 17: "olive_groves",
    18: "pastures", 19: "annual_crops_with_permanent_crops",
    20: "complex_cultivation_patterns", 21: "agriculture_with_natural_vegetation",
    22: "agro_forestry_areas", 23: "broad_leaved_forest",
    24: "coniferous_forest", 25: "mixed_forest", 26: "natural_grasslands",
    27: "moors_and_heathland", 28: "sclerophyllous_vegetation",
    29: "transitional_woodland_shrub", 30: "beaches_dunes_sands",
    31: "bare_rocks", 32: "sparsely_vegetated_areas", 33: "burnt_areas",
    34: "glaciers_and_perpetual_snow", 35: "inland_marshes", 36: "peat_bogs",
    37: "salt_marshes", 38: "salines", 39: "intertidal_flats",
    40: "water_courses", 41: "water_bodies", 42: "coastal_lagoons",
    43: "estuaries", 44: "sea_and_ocean",
}

print("Paths configured:")
print(f"  Source:      {TRAINING_DATA_DIR}")
print(f"  Destination: {PROCESSED_DIR}")
print(f"  Source exists: {TRAINING_DATA_DIR.exists()}")

Paths configured:
  Source:      /p/project1/training2600/sigursteinsson1/training_data_checkpoint
  Destination: /p/project1/training2600/sigursteinsson1/Final_Project/processed_data
  Source exists: True


In [28]:
# Load all splits from checkpoint 2. 60% | 20% | 20% split
data = {}
for split in ("train", "val", "test"):
    patches = np.load(TRAINING_DATA_DIR / f"patches_{split}.npz")["patches"]
    labels  = np.load(TRAINING_DATA_DIR / f"labels_{split}.npy")
    data[split] = {"patches": patches, "labels": labels}
    print(f"{split:>5}: {patches.shape}  labels shape: {labels.shape}")

# Count classes across all splits combined
all_labels = np.concatenate([data[s]["labels"] for s in ("train", "val", "test")])
class_counts = Counter(all_labels.tolist())

print(f"\nTotal patches: {len(all_labels):,}")
print(f"Classes found: {sorted(class_counts.keys())}")
print(f"\nClass counts (sorted by frequency):")
for cls_id, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    name = CORINE_CLASSES.get(int(cls_id), "unknown")
    print(f"  Class {int(cls_id):2d} ({name:<40}): {count:>6,}")

train: (20000, 3, 3, 16)  labels shape: (20000,)
  val: (10000, 3, 3, 16)  labels shape: (10000,)
 test: (10000, 3, 3, 16)  labels shape: (10000,)

Total patches: 40,000
Classes found: [2, 6, 7, 8, 11, 12, 18, 20, 21, 23, 24, 25, 27, 29, 31, 32, 35, 36, 40, 41]

Class counts (sorted by frequency):
  Class 24 (coniferous_forest                       ): 18,255
  Class 36 (peat_bogs                               ): 10,264
  Class 23 (broad_leaved_forest                     ):  3,032
  Class 25 (mixed_forest                            ):  2,743
  Class 29 (transitional_woodland_shrub             ):  2,595
  Class 27 (moors_and_heathland                     ):  1,439
  Class 41 (water_bodies                            ):  1,139
  Class 40 (water_courses                           ):    233
  Class 31 (bare_rocks                              ):     73
  Class 35 (inland_marshes                          ):     59
  Class  2 (discontinuous_urban_fabric              ):     33
  Class 32 (sparsel

In [29]:
# Drop any class with fewer than MIN_SAMPLES total across all splits
MIN_SAMPLES = 200

kept_classes = {
    cls_id for cls_id, count in class_counts.items()
    if count >= MIN_SAMPLES
}

print(f"Keeping {len(kept_classes)} classes (>= {MIN_SAMPLES} total samples):")
for cls_id in sorted(kept_classes):
    name = CORINE_CLASSES.get(int(cls_id), "unknown")
    total = class_counts[cls_id]
    print(f"  Class {int(cls_id):2d} ({name:<40}): {total:>6,}")

dropped = set(class_counts.keys()) - kept_classes
print(f"\nDropping {len(dropped)} rare classes: {sorted(dropped)}")

Keeping 8 classes (>= 200 total samples):
  Class 23 (broad_leaved_forest                     ):  3,032
  Class 24 (coniferous_forest                       ): 18,255
  Class 25 (mixed_forest                            ):  2,743
  Class 27 (moors_and_heathland                     ):  1,439
  Class 29 (transitional_woodland_shrub             ):  2,595
  Class 36 (peat_bogs                               ): 10,264
  Class 40 (water_courses                           ):    233
  Class 41 (water_bodies                            ):  1,139

Dropping 12 rare classes: [2, 6, 7, 8, 11, 12, 18, 20, 21, 31, 32, 35]


In [30]:
# TerraTorch NonGeoClassificationDataset expects:
#   processed_data/
#     train/class_name/*.npy
#     val/class_name/*.npy
#     test/class_name/*.npy
# Each .npy file is one patch: shape (C, H, W) = (16, 3, 3)

def build_terratorch_dataset(data, kept_classes, out_dir):
    out_dir = Path(out_dir)
    
    split_counts = {}
    for split in ("train", "val", "test"):
        patches = data[split]["patches"]  # (N, 3, 3, 16)
        labels  = data[split]["labels"]   # (N,)
        
        split_counts[split] = Counter()
        saved = 0
        
        for idx in range(len(patches)):
            cls_id = int(labels[idx])
            if cls_id not in kept_classes:
                continue
            
            cls_name = CORINE_CLASSES[cls_id]
            cls_dir  = out_dir / split / cls_name
            cls_dir.mkdir(parents=True, exist_ok=True)
            
            # Convert (3, 3, 16) → (16, 3, 3) for PyTorch channel-first convention
            patch_chw = patches[idx].transpose(2, 0, 1).astype(np.float32)
            
            np.save(cls_dir / f"{idx:06d}.npy", patch_chw)
            split_counts[split][cls_id] += 1
            saved += 1
        
        print(f"  {split:>5}: saved {saved:,} patches across "
              f"{len(split_counts[split])} classes")
    
    return split_counts

print("Building TerraTorch dataset layout...")
print(f"Output: {PROCESSED_DIR}\n")

if PROCESSED_DIR.exists():
    print("WARNING: output directory already exists — clearing it")
    shutil.rmtree(PROCESSED_DIR)

split_counts = build_terratorch_dataset(data, kept_classes, PROCESSED_DIR)
print("\nDone.")

Building TerraTorch dataset layout...
Output: /p/project1/training2600/sigursteinsson1/Final_Project/processed_data

  train: saved 19,838 patches across 8 classes
    val: saved 9,933 patches across 8 classes
   test: saved 9,929 patches across 8 classes

Done.


In [31]:
# Verify the folder layout looks correct
print("Folder structure verification:")
for split in ("train", "val", "test"):
    split_dir = PROCESSED_DIR / split
    classes_found = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    total_files = sum(
        len(list(d.glob("*.npy")))
        for d in split_dir.iterdir() if d.is_dir()
    )
    print(f"\n  {split} ({total_files:,} files, {len(classes_found)} classes):")
    for cls_name in classes_found:
        n = len(list((split_dir / cls_name).glob("*.npy")))
        print(f"    {cls_name:<45} {n:>5,}")

# Save metadata for reproducibility
metadata = {
    "source_dir": str(TRAINING_DATA_DIR),
    "output_dir": str(PROCESSED_DIR),
    "min_samples_threshold": MIN_SAMPLES,
    "kept_classes": {
        str(k): CORINE_CLASSES[k] for k in sorted(kept_classes)
    },
    "dropped_classes": sorted([int(x) for x in dropped]),
    "patch_shape_input":  "(N, 3, 3, 16)  — height x width x channels",
    "patch_shape_output": "(16, 3, 3)     — channels x height x width (PyTorch convention)",
    "splits": {
        split: dict(split_counts[split]) for split in ("train", "val", "test")
    }
}

meta_path = FINAL_PROJECT_DIR / "metadata_processed.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nMetadata saved to: {meta_path}")

Folder structure verification:

  train (19,838 files, 8 classes):
    broad_leaved_forest                           1,787
    coniferous_forest                             8,888
    mixed_forest                                  1,220
    moors_and_heathland                             713
    peat_bogs                                     5,190
    transitional_woodland_shrub                   1,362
    water_bodies                                    542
    water_courses                                   136

  val (9,933 files, 8 classes):
    broad_leaved_forest                             931
    coniferous_forest                             4,339
    mixed_forest                                    650
    moors_and_heathland                             475
    peat_bogs                                     2,616
    transitional_woodland_shrub                     533
    water_bodies                                    338
    water_courses                                    51

  t